In [11]:
# load packages
import numpy as np 
import scipy.io
from scipy.io   import  loadmat
import pandas as pd
import os
import matplotlib.pyplot as plt #import matplotlib as plt
import seaborn as sns #import mat73
import pickle as pkl
from datetime import datetime
from itertools import chain
from scipy.io import savemat
import re

In [12]:
from pathlib import Path
MainDir = Path.home() / "Desktop" / "Visual_Plasticty_Pipeline" / "Pipeline Processed Outputs" / "Spreadsheets"

# change to the directory
os.chdir(MainDir)  
d = os.listdir(MainDir)  # list files in dir
print(f'Available Files to choose from: {len(d)}')
print(f'Files on hand: {d}')

Available Files to choose from: 8
Files on hand: ['.Rhistory', '.DS_Store', 'old_data.csv', 'LongitudinalSSRP_PreS1_Table_RMS_Contrast_CRF_20260408_143823.mat', 'data.csv', '.RData', 'LongitudinalSSRP_PreS1_Table_RMS_Contrast_CRF_20250922_133009.mat', '.Rapp.history']


In [13]:
df = d[4]
# make full paths
file_path1 = os.path.join(MainDir, df)

# check file 1
print(f'\nFile #1: {file_path1}')
print('Does File #1 Exist?', os.path.exists(file_path1))

# Load the CSV
df = pd.read_csv(file_path1)
print(df.head())  # check the first few rows to confirm it loaded correctly


File #1: /Users/patricia.naomi/Desktop/Visual_Plasticty_Pipeline/Pipeline Processed Outputs/Spreadsheets/data.csv
Does File #1 Exist? True
   Sub_ID  Sess_Day  Attn_Dep  Hemifield  Sweep       Pre      Post   NL_ID
0     0.0       1.0       1.0        0.0    0.0  0.144275  0.097880  2651.0
1     0.0       1.0       1.0        0.0    1.0  0.338603  0.287478  2651.0
2     0.0       1.0       1.0        0.0    2.0  0.574862  0.574030  2651.0
3     0.0       1.0       1.0        0.0    3.0  0.829848  0.799320  2651.0
4     0.0       1.0       1.0        0.0    4.0  1.000000  0.954822  2651.0


## ***Post-Hoc Analysis of Sample Size Based on Power Analysis***

### **Power analysis will be performed on the main pre/post effect reported in Limon et al. 2026**

### 1) Overall Pre/Post Effect


### **1 | Pre/Post Within Subjects Design**

In [15]:
# Match exclusions from R analysis
exclude_ids = [8, 10, 20, 22, 26, 27, 28, 29, 30, 32, 33, 34]
BadSubjs = [-10]

df = pd.read_csv(file_path1)

# Create a lookup of Sub_ID to NL_ID before filtering
id_lookup = df.groupby('Sub_ID')['NL_ID'].first()

# Apply same exclusions as R
df_filtered = df[~df['Sub_ID'].isin(exclude_ids)]
df_filtered = df_filtered[~df_filtered['NL_ID'].isin(BadSubjs)]

# Get included and excluded NL_IDs
excluded_NL = id_lookup[id_lookup.index.isin(exclude_ids)].astype(int).tolist()
included_NL = sorted(df_filtered['NL_ID'].unique().astype(int).tolist())

print(f"Excluded Sub_IDs: {exclude_ids}")
print(f"Excluded NL_IDs:  {excluded_NL}")
print(f"N included: {df_filtered['Sub_ID'].nunique()}")

print(f"Included NL_IDs: {included_NL}")

Excluded Sub_IDs: [8, 10, 20, 22, 26, 27, 28, 29, 30, 32, 33, 34]
Excluded NL_IDs:  [2660, 2663, 2676, 2678, 2708, 2715, 2716, 2726, 2727, 2733, 2734, 2737]
N included: 26
Included NL_IDs: [2651, 2652, 2653, 2654, 2655, 2657, 2658, 2659, 2661, 2664, 2665, 2666, 2667, 2668, 2669, 2670, 2672, 2674, 2677, 2695, 2696, 2697, 2728, 345202, 345215, 345216]


In [ ]:
subject_stats = df_filtered.groupby('Sub_ID')[['Pre','Post']].mean()

print(f"Pre mean:  {subject_stats['Pre'].mean():.4f}")
print(f"Pre SD:    {subject_stats['Pre'].std():.4f}")
print(f"Post mean: {subject_stats['Post'].mean():.4f}")
print(f"Post SD:   {subject_stats['Post'].std():.4f}")

# Difference scores
subject_stats['diff'] = subject_stats['Post'] - subject_stats['Pre']

mean_diff = subject_stats['diff'].mean()
sd_diff = subject_stats['diff'].std()
correlation = subject_stats['Pre'].corr(subject_stats['Post'])

print(f"\nMean of difference:   {mean_diff:.4f}")
print(f"SD of difference:     {sd_diff:.4f}")
print(f"Pre-Post correlation: {correlation:.4f}")
print(f"N:                    {len(subject_stats)}")

Pre mean:             0.7127
Pre SD:               0.0594
Post mean:            0.7370
Post SD:              0.0821

Mean of difference:   0.0243
SD of difference:     0.0588
Pre-Post correlation: 0.6986
N:                    26


In [19]:
print(f"Based on G Power, Cohens dz is:  {0.41:.2f}")

Based on G Power, Cohens dz is:  0.41
